In [1]:
import os
import re
import requests
import wikipediaapi
import numpy as np
import json
from pymilvus import connections, FieldSchema, CollectionSchema, DataType, Collection, utility
from typing import List, Dict, Any
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from pathlib import Path
from sentence_transformers import SentenceTransformer
from datetime import datetime, timezone
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

#### Konfiguration

In [2]:
USER_AGENT = os.getenv(
    "USER_AGENT",
    "TourGuideAI/1.0 (Learning project) Mozilla/5.0 (Windows NT 10.0; Win64; x64)")
HEADERS = {"User-Agent": USER_AGENT}
DATA_WIKI = Path('../data/wiki')
DATA_JSON = Path('../data/ausflugziele')
BASE_URL_BOOTSVERLEIH = "https://www.spreewald-info.de"

# Embedding-Modell

EMBEDDING_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

# Milvus-Verbindung
connections.connect("default", host="127.0.0.1", port="19530")


#### 1. Quelle: Wikipedia

In [3]:
# 1. Wikipedia als Quelle

wiki = wikipediaapi.Wikipedia(language='de',
                             extract_format=wikipediaapi.ExtractFormat.WIKI,
                             user_agent=USER_AGENT)

# URL's laden
def load_urls(path: Path):
    urls = []
    for file in path.glob("*.json"):
        with open(file, "r", encoding="utf-8") as f:
            data = json.load(f)

            for entry in data:
                urls.append(entry["url"])
    return urls

# urls_wiki

# Wiki texte von unwichtige Teile bereinigen
def remove_references(text: str) -> str:
    stop_sections = ['Einzelnachweise', 'Weblinks', 'Literatur', 'Quellen', 'Fußnoten']
    for section in stop_sections:
        if section in text:
            text = text.split(section)[0]
    return text.strip()

def clean_wiki_text(text):
    text = remove_references(text)
     # Zeilenende normalisieren
    text = text.replace("\r\n", "\n")

    return text.strip()

#Titel aus URL extrahieren
def extract_title_from_url(url: str) -> str:
    match = re.search(r'/wiki/([^#]+)', url)
    if match:
        return match.group(1)
    return None

def load_wikipedia_text(title: str) -> dict:
    page = wiki.page(title)
    if not page.exists():
        return {'error': 'Page not found'}

    text = clean_wiki_text(page.text)
    return {
        'text': text,
        'title': page.title,
        'source': page.fullurl,
        'license': 'CC BY-SA 4.0'
    }




#
# wiki_data = {}
# for url in urls_wiki:
#     title = extract_title_from_url(url)
#     wiki_data[title] = load_wikipedia_text(title)


# def get_summary_from_wiki(text: str, n=2) -> str:
#     paragraphs = text.split('\n\n')
#     summary = ' '.join(paragraphs[:n])
#     return summary

def chunk_wiki_text(text: str, max_words: int = 200, overlap: int = 50):
    words = text.split()
    if not words:
        return []

    chunks = []
    start = 0

    while start < len(words):
        end = start + max_words
        chunk = " ".join(words[start:end])
        if chunk.strip():
            chunks.append(chunk)

        start = max(0, end - overlap)  # Überlappung für Kontext
        if start >= len(words):
            break

    return chunks


urls_wiki = load_urls(DATA_WIKI)
wiki_chunks = []

for url in urls_wiki:
    title = extract_title_from_url(url)
    if not title:
        continue

    data = load_wikipedia_text(title)
    if "text" not in data:
        continue

    chunks = chunk_wiki_text(data['text'], max_words=200, overlap=50)
    for i, chunk in enumerate(chunks):
            wiki_chunks.append({
                "text": chunk,
                "title": data['title'],
                "url": data['source'],
                "license": data['license']
            })
print(f"{len(wiki_chunks)} Text-Chunks mit Metadaten erstellt.")


# Collection-Schema definieren
collection_name = "wiki_collection"
if utility.has_collection(collection_name):
    utility.drop_collection(collection_name)
fields = [
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=384),
    FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=4096),
    FieldSchema(name="title", dtype=DataType.VARCHAR, max_length=1024),
    FieldSchema(name="url", dtype=DataType.VARCHAR, max_length=1024),
    FieldSchema(name="license", dtype=DataType.VARCHAR, max_length=64),
]
schema = CollectionSchema(fields, description="Wikipedia Chunks für semantische Suche")

collection = Collection(name=collection_name, schema=schema)

#Daten in Milvus einfügen
# Milvus generiert IDs automatisch
texts = [chunk['text'] for chunk in wiki_chunks]
titles = [chunk['title'] for chunk in wiki_chunks]
urls = [chunk['url'] for chunk in wiki_chunks]
licenses = [chunk['license'] for chunk in wiki_chunks]

wiki_embeddings = embedding_model.encode(texts).astype(np.float32).tolist()
collection.insert([wiki_embeddings, texts, titles, urls, licenses])


# print(f"Embeddings für {len(wiki_embeddings)} Wiki-Chunks erstellt.")



#Index erstellen (HNSW)
index_params = {
    "index_type": "HNSW",
    "metric_type": "COSINE",
    "params": {"M": 16, "efConstruction": 200}
}
collection.create_index(field_name="embedding", index_params=index_params)
print(f"Index für Collection '{collection_name}' erstellt.")

collection.load()
#Semantische Suche in Milvus
def semantic_search_milvus_wiki(query: str, k: int = 5):
    collection = Collection("wiki_collection")
    q_emb = embedding_model.encode([query]).astype(np.float32).tolist()
    results = collection.search(
        data=q_emb,
        anns_field="embedding",
        param={"metric_type": "COSINE", "params": {"ef": 50}},
        limit=k,
        output_fields=["text", "title", "url", "license"]
    )
    output = []
    for result in results:
        for hit in result:
            output.append({
                "score": hit.score,
                "text": hit.entity.get("text"),
                "title": hit.entity.get("title"),
                "url": hit.entity.get("url"),
                "license": hit.entity.get("license")
            })
    return output

# Test
# query = "Geschichte, Spreewald"
# results = semantic_search_milvus_wiki(query, k=5)
#
# for r in results:
#     print(f"\n--- Treffer: {r['title']} ---")
#     print(f"Score: {r['score']:.4f}")
#     print(f"URL: {r['url']}")
#     print(f"Text: {r['text'][:500]}...")  # Nur die ersten 300 Zeichen anzeigen


206 Text-Chunks mit Metadaten erstellt.
Index für Collection 'wiki_collection' erstellt.


#### 2. Eigene JSON-Datei

In [4]:
def load_json_data(path):
    data = []
    for filename in os.listdir(path):
        # Nur JSON-Dateien verarbeiten
        if not filename.endswith('.json'):
            continue
            # Öffnen und Laden der JSON-Datei im Lesemodus
        with open(os.path.join(path, filename), 'r', encoding='utf-8') as f:
                # Inhalt der JSON-Datei laden
            content = json.load(f)
            if isinstance(content, list):
                  data.extend(content)
            else:
                data.append(content)
    return data

#Collection erstellen
data = load_json_data(DATA_JSON)

# def clean_text(text):
#      # Kleinbuchstaben, Entfernen von Sonderzeichen
#     text = text.lower().replace(",", " ").replace("!", " ").replace("?", " ").replace("-", " ").replace(")", " ").strip()
#     text = text.split()
#     return text

docs_list = []

for i, doc in enumerate(data, start=1):
    beschreibung = doc.get('beschreibung')
    if not beschreibung:
        continue

    text = f"""
    Name: {doc.get('name')}
    Ort: {doc.get('ort')}
    Region: {doc.get('region')}
    Öffnungszeiten: {doc.get('oeffnungszeiten')}
    Eintrittspreise: {doc.get('eintrittspreise')}
    Zielgruppe: {doc.get('zielgruppe')}
    Kategorie: {doc.get('kategorie')}
    Beschreibung: {doc.get('beschreibung')}
    """.strip()

    docs_list.append({
        "name": doc.get("name") or "",
        "ort": doc.get("ort") or "",
        "region": doc.get("region") or "",
        "oeffnungszeiten": doc.get("oeffnungszeiten"),
        "text": text
    })

	# Vorschau ausgeben
#             print(f"\n--- Dokument {i}: {doc.get('name', 'Unbekannt')} ---")
#             print(f"\n{doc_entry['text'][:800]}...")  # nur die ersten 300 Zeichen
#
#         else:
#             print(f"Fehlende Beschreibung in Dokument: {doc.get('name', 'unknown')}")
# print(f"{len(docsListe)} Dokumente für RAG vorbereitet.")

#embedding
#model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
#model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
# texts = [doc['text'] for doc in docs_list]
#
#
# embeddings_json = embedding_model.encode(texts)
# embeddings = np.array(embeddings_json).astype('float32')

#Daten in Milvus einfügen


collection_name = "ausflug_collection"

if utility.has_collection(collection_name):
    utility.drop_collection(collection_name)

fields = [
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=384),
    FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=4096),
    FieldSchema(name="name", dtype=DataType.VARCHAR, max_length=512),
    FieldSchema(name="ort", dtype=DataType.VARCHAR, max_length=256),
    FieldSchema(name="region", dtype=DataType.VARCHAR, max_length=256)
]

schema = CollectionSchema(fields, description="Ausflugziele JSON Datenbank")

collection = Collection(collection_name, schema)

texts = [doc["text"] or "" for doc in docs_list]
names = [doc["name"] or "" for doc in docs_list]
orte = [doc["ort"] or "" for doc in docs_list]
regions = [doc["region"] or "" for doc in docs_list]

embeddings = embedding_model.encode(texts).astype("float32").tolist()

collection.insert([
    embeddings,
    texts,
    names,
    orte,
    regions
])

#Index erstellen
index_params = {

    "index_type": "HNSW",
    "metric_type": "COSINE",  #Ähnlichkeitsmaß für die Suche

    "params": {
        "M": 16,  # Anzahl der Verbindungen pro Knoten
        "efConstruction": 200  # Genauigkeit vs. Geschwindigkeit beim Indexaufbau
    }
}

collection.create_index(
    field_name="embedding",
    index_params=index_params
)

collection.load()

def semantic_search_json(query, k=5):
    collection = Collection("ausflug_collection")

    q_emb = embedding_model.encode([query]).astype("float32").tolist()

    results = collection.search(
        data=q_emb,
        anns_field="embedding",
        param={
            "metric_type": "COSINE",
            "params": {"ef": 50}
        },
        limit=k,
        output_fields=["text", "name", "ort", "region"]
    )

    output = []

    for hits in results:
        for hit in hits:
            output.append({
                "name": hit.entity.get("name"),
                "ort": hit.entity.get("ort"),
                "region": hit.entity.get("region"),
                "text": hit.entity.get("text"),
                "score": hit.score
            })

    return output


#### 3. Bootsverleih-Webseite (Web Scraping)
Hier werden die Preise für Bootsverleih extrahiert

In [5]:
# Links sammeln: Es gibt mehrere Seiten mit Bootsverleih, die alle unter "/paddeln/bootsverleih/" liegen.
def collect_bootsverleih_links():
    url = BASE_URL_BOOTSVERLEIH + "/paddeln/bootsverleih/"
    resp = requests.get(url, headers=HEADERS, timeout=20)
    resp.raise_for_status()
    html = resp.text
    soup = BeautifulSoup(html, "html.parser")

    links = []

    for a in soup.select("a"):  #alle Links auf der Seite durchgehen
        href = a.get("href")
        if not href:
            continue

        if href.startswith("/paddeln/bootsverleih/") and href != "/paddeln/bootsverleih/":
            links.append(urljoin(BASE_URL_BOOTSVERLEIH, href))
        # if href and "/paddeln/bootsverleih/" in href and href.count("/") >= 3:
        #     links.append(urljoin(BASE, href))

    links = list(set(links))  #Duplikate entfernen
    return links

# links = collect_bootsverleih_links()
# for i, link in enumerate(links, start=1):
#     print(f"{i}. {link}")


# Daten extrahieren
def parse_prices(url):
    resp = requests.get(url, headers=HEADERS, timeout=20)
    resp.raise_for_status()
    html = resp.text
    soup = BeautifulSoup(html, "html.parser")

    title_tag = soup.find("h1")
    title = title_tag.get_text(strip=True) if title_tag else "Unknown"

    prices = []

    for li in soup.select(".anbieterDetailsInfobox li"):
        text = " ".join(li.stripped_strings)
        if "Euro" in text or "€" in text:
            prices.append(text)

    return {
        "anbieter": title,
        "url": url,
        "preise": prices
    }

# Alle Anbieter und Preise sammeln. parce_prices() ist widerverwendbar
def collect_provider_and_prices():
    links = collect_bootsverleih_links()
    docs = []

    for link in links:
        try:
            data = parse_prices(link)
            docs.append(data)

            # print("\nAnbieter:", data["anbieter"])
            # print("URL:", data["url"])
            # for p in data["preise"]:
            #     print("  -", p)

        except Exception as e:
            print("Fehler bei:", link, e)

    print("\nGesamt Anbieter gescraped:", len(docs))
    return docs

#####################################################################

#JSON für RAG vorbereiten. Die Preise können sich ändern, deswegen werden nicht direckt in den Text stehen
def write_jsonl(docs, path: Path):
    #output_path = Path("providers.jsonl")
    retrieved_at = datetime.now(timezone.utc).isoformat()
    with path.open("w", encoding="utf-8") as f:
        for item in docs:
           #text feld für Embeddings vorbereiten

            anbieter = item.get("anbieter", "")
            source_url = item.get("url", "")

            text = (
                f"Anbieter: {anbieter}\n"
                f"Leistung: Bootsverleih\n"
                f"Hinweis: Aktuelle Preise bitte über die Quelle abrufen."
            ).strip()

            doc_entry = {
                "id": source_url,
                "text": text,
                "metadata": {
                    "source_url": source_url,
                    "anbieter": anbieter,
                    "license": "unknown",
                    "retrieved_at": retrieved_at,
                }
            }

            f.write(json.dumps(doc_entry, ensure_ascii=False) + "\n")

#####################################################################
docs = collect_provider_and_prices()
write_jsonl(docs, Path("providers.jsonl"))
# embedding
def load_bootsverleih_jsonl(path: Path):
    data = []

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)
            data.append(item)
    print(f"{len(data)} Dokumente geladen.")
    return data


data = load_bootsverleih_jsonl(Path("providers.jsonl"))

texts = [item['text'] for item in data]

embeddings_preise = embedding_model.encode(texts)
embeddings = np.array(embeddings_preise).astype('float32')
print(f"Embeddings für {len(embeddings)} Dokumente erstellt.")


#####################################################################
# Live Preise abrufen.

def get_bootsverleih_prices(docs: List[dict]) -> str:
    texts = []

    for d in docs:
        name = d["anbieter"]
        url = d["url"]

        # Preise echtzeitig abrufen
        live_data = parse_prices(url)
        prices = live_data.get("preise", [])

        price_text = "\n".join(f"- {p}" for p in prices) if prices else "Keine Preise gefunden."
        texts.append(f"Anbieter: {name}\nPreise:\n{price_text}\nQuelle: {url}")

    return "\n\n".join(texts)


Gesamt Anbieter gescraped: 17
17 Dokumente geladen.
Embeddings für 17 Dokumente erstellt.


#### 4. OpenStreetMap für Öffnungszeiten von Museen.

In [6]:
def search_place(query: str) -> Dict[str, Any]:
    url = "https://nominatim.openstreetmap.org/search"
    params = {
        "q": query,
        "format": "json",
        "limit": 1,
        "addressdetails": 1
    }

    r = requests.get(url, params=params, headers=HEADERS, timeout=10)
    r.raise_for_status()
    data = r.json()

    if not data:
        return {}

    return {
        "name": data[0]["display_name"],
        "lat": float(data[0]["lat"]),
        "lon": float(data[0]["lon"]),
        "city": data[0]["address"].get("city") or data[0]["address"].get("town"),
        "state": data[0]["address"].get("state"),
        "country": data[0]["address"].get("country")
    }


def get_museums_opening_hours(lat: float, lon: float, radius: int = 10000):
    query = f"""
    [out:json][timeout:25];

    (
      node["tourism"="museum"](around:{radius},{lat},{lon});
      way["tourism"="museum"](around:{radius},{lat},{lon});
      relation["tourism"="museum"](around:{radius},{lat},{lon});
    );

    out tags center;
    """

    r = requests.post(
        "https://overpass-api.de/api/interpreter",
        data=query,
        headers=HEADERS,
        timeout=60
    )

    r.raise_for_status()
    data = r.json()

    museums = []

    for el in data.get("elements", []):
        tags = el.get("tags", {})

        if not tags.get("name"):
            continue

        lat_val = el.get("lat") or el.get("center", {}).get("lat")
        lon_val = el.get("lon") or el.get("center", {}).get("lon")

        museums.append({
            "name": tags.get("name"),
            "opening_hours": tags.get("opening_hours"),
            "lat": lat_val,
            "lon": lon_val,
            "street": tags.get("addr:street"),
            "housenumber": tags.get("addr:housenumber"),
            "postcode": tags.get("addr:postcode"),
            "city": tags.get("addr:city")
        })

    return museums



#context für LLM+RAG

def get_osm_context(query: str):
    place = search_place(query)

    if not place:
        return f"Keine Informationen zu {query}"

    museums = get_museums_opening_hours(place["lat"], place["lon"])

    texts = []
    for m in museums:
        adress = " ".join(filter(None, [m.get("street"), m.get("housenumber"), m.get("postcode"), m.get("city")]))
        texts.append(
            f"Name: {m.get('name')}\n"
            f"Öffnungszeiten: {m.get('opening_hours') or 'Keine Informationen verfügbar'}\n"
            f"Adresse: {adress}\n"
        )
    return "\n\n".join(texts)

#### 5. Zusammenführung aller Quellen für RAG

In [7]:
def build_context_for_RAG(query: str, k: int = 5):

    results = []

    # Wikipedia
    wiki_results = semantic_search_milvus_wiki(query, k=k)
    for r in wiki_results:
        results.append({
            "text": r["text"],
            "source": r["url"],
            "score": r["score"]
        })

    # Eigene JSON-Daten
    json_results = semantic_search_json(query, k=k)
    for r in json_results:
        results.append({
            "text": r["text"],
            "source": f"{r['name']} ({r['ort']})",
            "score": r["score"]
        })

    # Bootsverleih (Live Preise). Preise werden nur bei passenden Qieries gegeben
    if "boot" in query.lower() or "paddeln" in query.lower() or "kahn" in query.lower():
        docs = collect_provider_and_prices()
        boots_text = get_bootsverleih_prices(docs)
        if isinstance(boots_text, str) and boots_text.strip():
            results.append({
                "text": boots_text,
                "source": "https://www.spreewald-info.de/paddeln/bootsverleih/",
                "score": 0.3,
            })

    # OSM Museen
    if "museum" in query.lower() or "öffnung" in query.lower() or "zeit" in query.lower():
        osm_context = get_osm_context(query)

        if osm_context.strip():
            results.append({
                "text": f"[OSM Museen]\n{osm_context}",
                "source": "OpenStreetMap (Nominatim/Overpass)",
                "score": 0.35,
            })

    results = sorted(results, key=lambda x: x["score"], reverse=True)

    selected = results[:k]


    context = "\n\n".join([r["text"] for r in selected])
    # Quellen deduplizieren, Reihenfolge nach Score beibehalten
    sources = list(dict.fromkeys([r["source"] for r in selected]))
    return context, sources

#### 6. Antwortgenerierung mit LLM

In [8]:
# os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    dtype="auto"
)

generator = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,

)


Device set to use cpu


In [9]:
# # Annahme: tokenizer ist bereits geladen
# max_model_tokens = tokenizer.model_max_length  # z.B. 512
#
# def trim_context_to_fit(context: str, question: str, tokenizer, reserve_for_question=128):
#     # reserve_for_question: Tokens, die für die Frage + Instruktion frei bleiben
#     max_context_tokens = max_model_tokens - reserve_for_question
#     # Tokenize context
#     ctx_tokens = tokenizer.encode(context, truncation=False)
#     if len(ctx_tokens) <= max_context_tokens:
#         return context
#     # Wenn zu lang: nimm die ersten max_context_tokens Tokens (oder letzten, je nach Präferenz)
#     trimmed_tokens = ctx_tokens[:max_context_tokens]
#     trimmed_text = tokenizer.decode(trimmed_tokens, skip_special_tokens=True, clean_up_tokenization_spaces=True)
#     return trimmed_text
#
# # Beispiel in generate_answer einbauen
# def generate_answer(query: str, k: int = 5):
#     context, source_url = build_context_for_RAG(query, k=k)
#     # Kontext trimmen
#     safe_context = trim_context_to_fit(context, query, tokenizer, reserve_for_question=128)
#
#     prompt = f"""
# Du bist ein Reiseassistent für das Region Spreewald.
#
# Beantworte die Frage nur anhand des gegebenen Kontexts.
# Wenn du keine Antwort auf die Frage findest, sagt das ehrlich. Du darfst nicht fantasieren oder
# nicht existierende Orte, Preise oder Öffnungszeiten nennen. Du musst immer die Quellen angeben, auf
# denen deine Antwort basiert. Du darfst auch die Quellen nicht erfinden.
#
# Kontext:
# {safe_context}
#
# Frage:
# {query}
#
# Antwort:
# """
#
#     out = generator(prompt, do_sample=True, temperature=0.9, max_new_tokens=256)
#     answer = out[0]["generated_text"].strip() if isinstance(out, list) else str(out)
#     return {"answer": answer, "sources": list(dict.fromkeys(source_url))}


In [10]:

def trim_context_to_fit(context: str, question, tokenizer, max_tokens=tokenizer.model_max_length, reserve_for_question: int=128):
    question_tokens = tokenizer.encode(question)
    max_context_tokens = max_tokens - len(question_tokens)-reserve_for_question
    context_tokens = tokenizer.encode(context, truncation=False)

    if len(context_tokens) <= max_context_tokens:
        return context

    trimmed_tokens = context_tokens[:max_context_tokens]

    return tokenizer.decode(trimmed_tokens, skip_special_tokens=True)

In [14]:
# def generate_llm(prompt: str, model, tokenizer):
#     inputs = tokenizer(prompt, return_tensors="pt")
#
#     output = model.generate(
#         **inputs,
#         max_new_tokens=200,
#         do_sample=False
#     )
#     return tokenizer.decode(output[0], skip_special_tokens=True)

def generate_answer(query: str, tokenizer, k: int = 5):

    context, sources = build_context_for_RAG(query, k=k)
    context = trim_context_to_fit(context, query, tokenizer)
    prompt = f"""
Du bist ein Reiseassistent für das Region Spreewald.

Beantworte die Frage nur anhand des gegebenen Kontexts.
Wenn du keine Antwort auf die Frage findest, sagt das ehrlich. Du darfst nicht fantasieren oder
nicht existierende Orte, Preise oder Öffnungszeiten nennen. Du musst immer die Quellen angeben, auf
denen deine Antwort basiert. Du darfst auch die Quellen nicht erfinden.

Kontext:
{context}

Frage:
{query}

Antwort:
"""

    answer = generator(prompt)[0]["generated_text"].strip()

    return {
        "answer": answer,
        "sources": sources }


In [15]:
#Test
query = "welche Museen kann ich montags besuchen? Mach mir eine Liste"
results = generate_answer(query, tokenizer)

print("Antwort:\n", results["answer"])
print("\nQuellen:")
for src in results["sources"]:
    print("- ", src)

Antwort:
 Name: Freilandmuseum Lehde Ort: Lübbenau, Luebbenau Region: Spreewald, Brandenburg ffnungszeiten: täglich 10:00 - 18:00 Uhr Eintrittspreise: Erwachsene: 6 €, Kinder unter 18 Jahren: frei Zielgruppe: Erwachsene, Jugendliche, Kinder Kategorie: Museum Description: Freilandmuseum Lehde ist ein Freilichtmuseum, das die traditionelle Lebensweise und Kultur der Spreewaldregion zeigt. Kunst, sind unter anderem Werke von Wolf Vostell, Emmett Williams, Christo, Niki de Saint Phalle. On the rbb-Gelände in Babelsberg befindet sich ein Standort des Deutschen Rundfunkarchivs (DRA). Am Park Sanssouci befindet sich das Mühlenmuseum in der Historischen Mühle, mit mühlenkundlicher Ausstellung und praktischer Darstellung des Mahlvorgangs. Kai Desinger öffnete im April 2012 mit Garage du Pont a mixture aus restaurant and automuseum. In the Räumen einer ehemaligen Tankstelle are some old cars ausgestellt, wobei der Schwerpunkt bei französischen Klassikern liegt. Ende 2019 wurde der Betrieb vorübe

In [17]:
#Test
query = "welche Oeffnungszeiten hat Freilandmuseum Lehde"
results = generate_answer(query, tokenizer)
print("Antwort:\n", results["answer"])
print("\nQuellen:")
for src in results["sources"]:
    print("- ", src)

Antwort:
 Freilandmuseum Lehde Ort: Lübbenau, Luebbenau Region: Spreewald, Brandenburg ffnungszeiten: täglich 10:00 - 18:00 Uhr Eintrittspreise: Erwachsene: 6 €, Kinder unter 18 Jahren: frei Zielgruppe: Erwachsene, Jugendliche, Kinder Kategorie: Museum Description: Freilandmuseum Lehde ist ein Freilichtmuseum, das die traditionelle Lebensweise und Kultur der Spreewald region shows. Visitors can historical buildings, Handwerksbetriebe und landwirtschaftliche Einrichtungen besichtigen, um an Einblick in the life of people in the region to receive. Kunst, sind unter anderem Werke von Wolf Vostell, Emmett Williams, Christo, Niki de Saint Phalle. On the rbb-Gelände in Babelsberg befindet sich ein Standort des Deutschen Rundfunkarchivs (DRA). Am Park Sanssouci ...

Quellen:
-  Freilandmuseum Lehde (Lübbenau, Luebbenau)
-  https://de.wikipedia.org/wiki/Potsdam
-  Bootsverleih Dolzke-Insel ()
-  Weißgerbermuseum (Doberlug-Kirchhain)
-  Heimatmuseum Dissen (Dissen-Striesow)
